In [15]:
!pip install transformers torch feedparser

In [16]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime
from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.nn.functional as F

import feedparser

# 1. SETUP & CONFIGURATION
# Note: To use real news, you would sign up for a NewsAPI or Twitter API key.
NEWS_API_KEY = "" # Add your API key here
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"


In [ ]:
# 1. SETUP & CONFIGURATION
# List of RSS feeds (common ones for EPL/FPL news)
RSS_FEEDS = [
    "https://www.theguardian.com/football/premierleague/rss",
    "https://api.foxsports.com/v1/rss?partnerKey=zBaFxv9pSqsE9uOfpS9u9pSqsE9uOfpS&tag=soccer",
    "https://brentfordfc.com/news/rss" # Example club specific feed
]

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

def fetch_rss_headlines(player_name):
    """
    Scans specified RSS feeds for mentions of a player name.
    """
    print(f"--- Scanning RSS Feeds for {player_name} ---")
    found_headlines = []

    for url in RSS_FEEDS:
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                # Check if player name exists in title or summary
                content_to_check = (entry.title + " " + entry.get('summary', '')).lower()
                if player_name.lower() in content_to_check:
                    found_headlines.append(entry.title)
                    print(f"Found mention in feed {url}: {entry.title}")
        except Exception as e:
            print(f"Error parsing {url}: {e}")

    # Fallback to mock data if no real news found (for demonstration)
    if not found_headlines:
        mock_data = {
            "Mohamed Salah": ["Salah expected to start against Arsenal."],
            "Erling Haaland": ["Haaland clinical in training, Pep says he is 'feeling great'."],
            "Bukayo Saka": ["Arteta hopeful on Saka fitness but 'will wait and see'."]
        }
        return mock_data.get(player_name, ["No major news updates for this player."])

    return found_headlines
# 3. MOCK FPL STATS DATASET
def get_fpl_base_stats():
    """
    Creates a dataframe of standard FPL metrics.
    """
    data = {
        'player_name': ['Mohamed Salah', 'Erling Haaland', 'Bukayo Saka', 'Kevin De Bruyne', 'Son Heung-min'],
        'xG': [0.85, 1.10, 0.45, 0.20, 0.55],
        'xA': [0.30, 0.15, 0.40, 0.75, 0.25],
        'ict_index': [12.5, 15.2, 9.8, 11.2, 10.5],
        'fixture_difficulty': [2, 4, 3, 2, 5],
        'actual_points': [8, 2, 7, 5, 4]
    }
    return pd.DataFrame(data)

# 4. FULL PIPELINE EXECUTION
def run_pipeline():
    # Step A: Get base FPL stats
    df = get_fpl_base_stats()

    # Step B: Enrich with Sentiment Data from RSS
    sentiment_list = []
    for name in df['player_name']:
        headlines = fetch_rss_headlines(name)
        print(f"Headlines for {name}: {headlines}")

run_pipeline()

--- Scanning RSS Feeds for Mohamed Salah ---
Headlines for Mohamed Salah: ['Salah expected to start against Arsenal.']
--- Scanning RSS Feeds for Erling Haaland ---
Found mention in feed https://www.theguardian.com/football/premierleague/rss: Brighton’s Kaoru Mitoma punishes Manchester City as title bid falters again
Headlines for Erling Haaland: ['Brighton’s Kaoru Mitoma punishes Manchester City as title bid falters again']
--- Scanning RSS Feeds for Bukayo Saka ---
Headlines for Bukayo Saka: ["Arteta hopeful on Saka fitness but 'will wait and see'."]
--- Scanning RSS Feeds for Kevin De Bruyne ---
Headlines for Kevin De Bruyne: ['No major news updates for this player.']
--- Scanning RSS Feeds for Son Heung-min ---
Headlines for Son Heung-min: ['No major news updates for this player.']
